# Module 2: Train and Evaluate a Keras-Based Classifier

> **Assignment (12 points):** Build, compile and train a CNN for agricultural vs non-agricultural land classification with Keras.

In [ ]:
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
PROJECT_ROOT = pathlib.Path(os.path.abspath(os.getcwd()))
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'images_dataSAT').is_dir():
        PROJECT_ROOT = p
        break
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42)
dataset_path = './images_dataSAT'
print('TensorFlow version:', tf.__version__)

### Task 1: Recursively walk `dataset_path` with `os.walk` to create `fnames` (all image files)

In [ ]:
fnames = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            fnames.append(os.path.join(root, f))

print('Total image files:', len(fnames))
print(fnames[:3])

### Task 2: Create `validation_generator` from `dataset_path`

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1. / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
)

train_generator = datagen.flow_from_directory(
    dataset_path, target_size=(64, 64), batch_size=32, class_mode='binary',
    subset='training', shuffle=True, seed=42)

validation_generator = datagen.flow_from_directory(
    dataset_path, target_size=(64, 64), batch_size=32, class_mode='binary',
    subset='validation', shuffle=False, seed=42)

### Task 3: Count the total number of layers in this CNN model

In [ ]:
def build_cnn(input_shape=(64, 64, 3)):
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])

model = build_cnn()
print('Total number of layers in the CNN model:', len(model.layers))

### Task 4: Create and compile `test_model` — four Conv2D layers and five Dense layers

In [ ]:
test_model = build_cnn()

conv_count   = sum(isinstance(l, tf.keras.layers.Conv2D) for l in test_model.layers)
dense_count  = sum(isinstance(l, tf.keras.layers.Dense) for l in test_model.layers)
print('Conv2D layers:', conv_count, '| Dense layers:', dense_count,
      '| Total layers:', len(test_model.layers))

test_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
test_model.summary()

### Task 5: Create the checkpoint callback for the model with maximum accuracy

In [ ]:
os.makedirs('models', exist_ok=True)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='models/best_cnn.keras',
    monitor='val_accuracy',     # keep the model with the highest accuracy
    mode='max',
    save_best_only=True,
    verbose=1,
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True)

## Train the model

In [ ]:
history = test_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5,
    steps_per_epoch=len(train_generator),
    callbacks=[checkpoint, early_stopping],
    verbose=1,
)

### Task 6: Plot the graph for training loss and validation loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('models/keras_cnn_loss.png')
plt.show()

## Save the trained model (reused by Module 3) and evaluate quickly

In [ ]:
test_model.save('models/cnn_model.keras')
print('Model saved -> models/cnn_model.keras')

validation_generator.reset()
y_true = validation_generator.classes
y_pred = (test_model.predict(validation_generator, verbose=0).flatten() > 0.5).astype(int)

from sklearn.metrics import accuracy_score, f1_score
print('Validation accuracy:', round(accuracy_score(y_true, y_pred), 4))
print('Validation F1     :', round(f1_score(y_true, y_pred), 4))